### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

BLOG : 
https://medium.com/@sahin.samia/query-expansion-in-enhancing-retrieval-augmented-generation-rag-d41153317383

In [1]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# step1 : Load and split the dataset
loader = TextLoader("langchain-crewai-dataset.txt")
raw_docs = loader.load()

# split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain-crewai-dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain-crewai-dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain-crewai-dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# step 2: Embedding
embedding_model = OpenAIEmbeddings()

# vectorstore
vectorstore = FAISS.from_documents(chunks, embedding_model)

# step 3: MMR Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

retriever


c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023559E2BA10>, search_type='mmr', search_kwargs={'k': 5})

In [3]:
# step 4 : LLM 
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023559E2ABA0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023559E29E80>, root_client=<openai.OpenAI object at 0x0000023559E71BD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000023559E72AD0>, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [4]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# step 5 : Prompt for Query expansion
query_expansion_prompt = PromptTemplate.from_template(
"""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"
Expanded query:
"""
)

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()

query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\nExpanded query:\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023559E2ABA0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023559E29E80>, root_client=<openai.OpenAI object at 0x0000023559E71BD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000023559E72AD0>, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [ ]:
# Example Test query expansion
query = {"query": "Langchain memory"}

result = query_expansion_chain.invoke(query)
result

'"Langchain memory" refers to the use of Langchain technology in computer memory systems. Langchain technology is a novel approach that leverages blockchain concepts to enhance memory performance, reliability, and security. Other terms related to Langchain memory include blockchain memory, distributed memory systems, decentralized memory architecture, and blockchain-based memory solutions. By incorporating Langchain technology into memory systems, organizations can benefit from improved data integrity, decentralization of memory storage, and resistance to unauthorized access or tampering.'

# RAG Pipeline

In [6]:
from langchain.chains.combine_documents import create_stuff_documents_chain

# step 6: RAG answering prompt
answer_prompt = PromptTemplate.from_template(
"""
Answer the question based on the context below.

Context: {context}

Question: {input}
"""
)

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=answer_prompt
)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext: {context}\n\nQuestion: {input}\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023559E2ABA0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023559E29E80>, root_client=<openai.OpenAI object at 0x0000023559E71BD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000023559E72AD0>, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [7]:
from langchain_core.runnables import RunnableMap

# Step 7: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    
    | document_chain
)

rag_pipeline

{
  input: RunnableLambda(...),
  context: RunnableLambda(...)
}
| RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
    context: RunnableLambda(format_docs)
  }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
  | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext: {context}\n\nQuestion: {input}\n')
  | ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000023559E2ABA0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000023559E29E80>, root_client=<openai.OpenAI object at 0x0000023559E71BD0>, root_async_client=<openai.AsyncOpenAI object at 0x0000023559E72AD0>, model_kwargs={}, openai_api_key=SecretStr('**********'))
  | StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [8]:
# Step 8: Run query
query = {"input": "What types of memory does LangChain support?"}

# query expansion
print(query_expansion_chain.invoke({"query":query}))

# RAG pipeline
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

{'input': 'What types of memory does LangChain support?',
 'expanded': 'Which memory types are compatible with LangChain? What are the supported memory technologies in LangChain? What memory options does LangChain work with?'}
✅ Answer:
 LangChain supports memory modules like ConversationBufferMemory and ConversationSummaryMemory.


In [9]:
# Step 8: Run query
query = {"input": "CrewAI agents?"}

# query expansion
print(query_expansion_chain.invoke({"query":query}))

# RAG pipeline
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

{'input': 'What are the abilities of CrewAI agents? Features of CrewAI agents? Details about CrewAI agents?'}
✅ Answer:
 CrewAI agents are structured into organized crews with defined roles such as researcher, planner, or executor, and they work together to complete tasks by dividing responsibilities, sharing context, and communicating with one another dynamically within a collaborative context.
